# Time-to-event prediction with C-MAPSS

Time-to-event prediction is the broad problem of estimating when a defined
event will occur. Here the event is engine failure and elapsed time is measured
in operating cycles.

## Learning goals

- define the event and time origin explicitly;
- construct cycles-to-failure with Polars;
- evaluate a regression baseline with an engine-level split.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
pdmdata.download("cmapss")
data = pdmdata.load("cmapss", subset="FD001", split="train")
data = data.with_columns(
    (pl.col("cycle").max().over("unit_number") - pl.col("cycle"))
    .alias("time_to_event_cycles"),
    pl.lit(True).alias("event_observed"),
)
data.select("unit_number", "cycle", "time_to_event_cycles", "event_observed").head()

In [ ]:
plt.hist(data["time_to_event_cycles"].to_numpy(), bins=60)
plt.xlabel("Operating cycles until failure")
plt.ylabel("Observations")
plt.show()

In [ ]:
unit_numbers = data["unit_number"].unique().sort().to_numpy()
split_index = int(len(unit_numbers) * 0.8)
training_units = unit_numbers[:split_index]
test_units = unit_numbers[split_index:]
train = data.filter(pl.col("unit_number").is_in(training_units))
test = data.filter(pl.col("unit_number").is_in(test_units))
features = [
    column for column in data.columns
    if column.startswith(("operation_", "sensor_"))
]

model = HistGradientBoostingRegressor(loss="absolute_error", random_state=0)
model.fit(train.select(features).to_numpy(), train["time_to_event_cycles"].to_numpy())
prediction = np.maximum(model.predict(test.select(features).to_numpy()), 0)
print(
    "Engine-held-out MAE:",
    f"{mean_absolute_error(test['time_to_event_cycles'].to_numpy(), prediction):.1f} cycles",
)

## Time-to-event versus survival analysis

This regression uses only observed failures and predicts one point estimate.
Survival analysis instead estimates an event-time distribution and can include
engines whose failure is not observed before data collection ends. See the
survival-analysis notebook for that formulation.